In [1]:
%cd .

d:\github\ssd


In [2]:
from config import SSDConfig
from model.ssd import SSD

In [3]:
config = SSDConfig(
    num_classes=21,
    image_size=300,
    feature_map_resolutions=[38, 19, 10, 5, 3, 1],
    box_scales=[0.1, 0.2, 0.37, 0.54, 0.71, 0.88],
    box_aspect_ratios=[
        [1, 2, 1/2], [1, 2, 1/2, 3, 1/3], 
        [1, 2, 1/2, 3, 1/3], [1, 2, 1/2, 3, 1/3],
        [1, 2, 1/2], [1, 2, 1/2]
    ],
    box_variances=[0.1, 0.2],
    box_clip=True,
    backbone_architecture=[64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'C', 512, 512, 512, 'M', 512, 512, 512],
    backbone_in_channels=3,
    backbone_batch_norm=False,
    l2_norm_channels=512,
    l2_norm_scale=20,
    extras_architecture=[256, 'S', 512, 128, 'S', 256, 128, 256, 128, 256],
    extras_in_channels=1024,
    extras_batch_norm=False,
    num_boxes=[4, 6, 6, 6, 4, 4]
)

In [4]:
model = SSD(config)

In [5]:
import torch

x = torch.rand(size=(1, 3, 300, 300))
x.shape

torch.Size([1, 3, 300, 300])

In [6]:
output = model(x)

VGG [torch.Size([1, 512, 38, 38]), torch.Size([1, 1024, 19, 19])]
Extra [torch.Size([1, 512, 38, 38]), torch.Size([1, 1024, 19, 19]), torch.Size([1, 512, 10, 10]), torch.Size([1, 256, 5, 5]), torch.Size([1, 256, 3, 3]), torch.Size([1, 256, 1, 1])]
6


In [74]:
ckpt = torch.load('./weights/checkpoint.pth')

In [75]:
vgg_state_dict = {
    k: v
    for k, v in ckpt.items()
    if k.startswith("vgg.")
}

new_state_dict = {}

for k, v in vgg_state_dict.items():
    new_key = k.replace("vgg.", "layers.")
    new_state_dict[new_key] = v
    
model.backbone.load_state_dict(new_state_dict, strict=False)

<All keys matched successfully>

In [76]:
l2_norm_state_dict = {
    k: v
    for k, v in ckpt.items()
    if k.startswith("l2_norm.")
}
l2_norm_state_dict["scale_weights"] = l2_norm_state_dict.pop("l2_norm.weight")

model.l2_norm.load_state_dict(l2_norm_state_dict, strict=False)

<All keys matched successfully>

In [77]:
extra_layers_state_dict = {
    k: v
    for k, v in ckpt.items()
    if k.startswith("extra_layers.")
}

new_state_dict = {}

cnt = 0
for idx, (k, v) in enumerate(extra_layers_state_dict.items()):
    new_key = k.replace(f"extra_layers.{cnt}", f"layers.{cnt * 2}")
    new_state_dict[new_key] = v
    if idx % 2 == 1:
        cnt += 1
    
model.extras.load_state_dict(new_state_dict, strict=False)

<All keys matched successfully>

In [78]:
conf_state_dict = {
    k: v
    for k, v in ckpt.items()
    if k.startswith("conf.")
}

new_state_dict = {}

for k, v in conf_state_dict.items():
    new_key = k.replace("conf.", "")
    new_state_dict[new_key] = v
    
model.head.conf_layers.load_state_dict(new_state_dict, strict=False)

<All keys matched successfully>

In [81]:
loc_state_dict = {
    k: v
    for k, v in ckpt.items()
    if k.startswith("loc.")
}

new_state_dict = {}

for k, v in loc_state_dict.items():
    new_key = k.replace("loc.", "")
    new_state_dict[new_key] = v
    
model.head.loc_layers.load_state_dict(new_state_dict, strict=False)

<All keys matched successfully>

In [87]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image = cv2.imread('./sample/test.jpg', cv2.IMREAD_COLOR)
rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

x = cv2.resize(image, (300, 300)).astype(np.float32)
x -= (104.0, 117.0, 123.0)
x = x.astype(np.float32)
x = x[:, :, ::-1].copy()
x = torch.from_numpy(x).permute(2, 0, 1)
x = x.unsqueeze(0) 

In [93]:
model(x)

VGG [torch.Size([1, 512, 38, 38]), torch.Size([1, 1024, 19, 19])]
Extra [torch.Size([1, 512, 38, 38]), torch.Size([1, 1024, 19, 19]), torch.Size([1, 512, 10, 10]), torch.Size([1, 256, 5, 5]), torch.Size([1, 256, 3, 3]), torch.Size([1, 256, 1, 1])]
6


(tensor([[[-0.1727, -0.2350, -3.2430, -2.9372],
          [ 0.0531, -0.1676, -1.1556, -0.4660],
          [ 0.0752, -0.6451, -4.2246, -3.5278],
          ...,
          [ 0.2896,  0.4902, -1.9317, -1.5756],
          [ 0.2997,  0.8361, -2.3563,  0.4196],
          [ 0.4548,  0.5820,  0.1154, -2.0073]]], grad_fn=<CatBackward0>),
 tensor([[[  8.0983,   0.5986,  -0.6890,  ...,  -1.5619,  -0.8326,  -0.6507],
          [  8.5966,   0.2221,  -0.6278,  ...,  -0.8627,  -0.2722,  -0.2671],
          [  8.3248,   1.3872,  -1.2909,  ...,  -1.6470,  -1.0596,  -1.0104],
          ...,
          [ 13.8673,   3.6558,   5.2894,  ...,  -2.9220,  -0.1742, -13.4250],
          [ 11.5817,   4.3134,   5.5749,  ...,  -2.6330,   0.5370, -11.4584],
          [  9.1432,   4.1318,   5.5527,  ...,  -1.7642,  -0.1087, -12.8010]]],
        grad_fn=<CatBackward0>),
 tensor([[0.0132, 0.0132, 0.1414, 0.1414],
         [0.0132, 0.0132, 0.1000, 0.1000],
         [0.0132, 0.0132, 0.1414, 0.0707],
         ...,
         

In [95]:
model.load_state_dict(
    torch.load('./weights/best.pt')
)

<All keys matched successfully>

In [ ]:
from typing import List

def ssd_inference(
    loc: torch.Tensor,
    conf: torch.Tensor,
    default_boxes: torch.Tensor,
    num_classes: int,
    background_label: int,
    top_k: int,
    conf_threshold: float,
    iou_threshold: float,
    variance: List[float],
) -> torch.Tensor:
    batch_size = loc.size(0)
    conf_preds = conf.transpose(2, 1)
    output = torch.zeros(batch_size, num_classes, top_k, 5, device=loc.device)

    for b in range(batch_size):
        decoded_boxes = decode_offsets_to_boxes(loc[b], default_boxes, variance)
        scores_per_class = conf_preds[b]

        for cls_id in range(num_classes):
            if cls_id == background_label:
                continue

            scores = scores_per_class[cls_id]
            mask = scores > conf_threshold

            if mask.sum() == 0:
                continue

            cls_scores = scores[mask]
            cls_boxes = decoded_boxes[mask]

            indices, count = nms(
                boxes=cls_boxes,
                scores=cls_scores,
                overlap=iou_threshold,
                top_k=top_k,
            )

            keep = indices[:count]
            output[b, cls_id, :count] = torch.cat(
                [cls_scores[keep].unsqueeze(1), cls_boxes[keep]],
                dim=1,
            )

    return output
